# 06 - Boundary/Raster Compatibility Check

This notebook checks whether the Mozambique ADM2 boundary layer and the WorldPop under-18 raster are spatially compatible before raster-to-vector aggregation.

**Why this matters:** zonal statistics only make sense when the vector polygons and raster grid describe overlapping places in the same coordinate reference system.

**Inputs**

- `data/raw/boundaries/geoBoundaries-MOZ-ADM2.geojson`
- `data/raw/worldpop/moz_under_age_18_2019/moz_T_Under_18_2019_CN_100m_R2025A_v1.tif`

**Outputs**

- Printed CRS comparison
- Printed boundary/raster extent comparison
- A Boolean check showing whether the two spatial extents intersect


## 1. Setup

Load the boundary and raster paths from the shared ChildReach path module. This keeps the notebook independently runnable while avoiding repeated hard-coded paths.


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import geopandas as gpd
import rasterio
from shapely.geometry import box

from childreach.paths import ADM2_BOUNDARIES_GEOJSON, WORLDPOP_UNDER18_TOTAL_2019_TIF

boundaries = gpd.read_file(ADM2_BOUNDARIES_GEOJSON)

print("Boundary CRS:", boundaries.crs)
print("Boundary bounds:")
print(boundaries.total_bounds)  # minx, miny, maxx, maxy

with rasterio.open(WORLDPOP_UNDER18_TOTAL_2019_TIF) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds
    print("Raster CRS:", raster_crs)
    print("Raster bounds:")
    print(raster_bounds)

boundary_box = box(*boundaries.total_bounds)
raster_box = box(*raster_bounds)

print("CRS match:", str(boundaries.crs) == str(raster_crs))
print("Bounds intersect:", boundary_box.intersects(raster_box))


Boundary CRS: EPSG:4326
Boundary bounds:
[ 30.21172507 -26.86704076  40.84040923 -10.47366563]
Raster CRS: EPSG:4326
Raster bounds:
BoundingBox(left=30.215832492469996, bottom=-26.867499556530017, right=40.839999116639994, top=-10.474166288770018)
CRS match: True
Bounds intersect: True


## 2. Interpret the compatibility checks

For this milestone, the important checks are:

1. **CRS match** — both layers should report the same coordinate reference system before aggregation.
2. **Bounds intersect** — the raster extent must overlap the ADM2 boundary extent.

A matching CRS and intersecting bounds do not prove the analysis is complete, but they confirm that the two spatial layers are aligned well enough to proceed to zonal statistics.

Note: EPSG:4326 stores longitude/latitude coordinates in degrees, not meters. That is fine for overlay and extraction checks here, but we should avoid describing this CRS as meter-based.


## Reproducibility handoff

If these checks pass, the next notebook can summarize raster values inside each ADM2 polygon. If either check fails, stop and diagnose the CRS, file choice, or spatial extent before running zonal statistics.
